# Test: Can vLLM serve Qwen/Qwen3.5-9B?

Standalone compatibility check, separate from `muspsy_serve_vllm.ipynb` so it can't disturb the already-working Llama-3 pipeline. This does **not** fine-tune anything - it just tries to serve the base `Qwen/Qwen3.5-9B` checkpoint (a vision-language model, but tested here as a pure-text chat endpoint, matching how MusPsy would actually use it) via vLLM and sends one test request.

Why this matters: `Qwen/Qwen3.5-9B` uses a newer hybrid attention architecture (`Qwen3_5ForConditionalGeneration`), and vLLM's GitHub history shows several "model type qwen3_5 not recognized" / weight-loading / KV-cache bugs landing as late as May 2026. `vllm==0.9.2` (pinned elsewhere in this project specifically to dodge a *different*, unrelated Llama-3 bug) almost certainly predates `qwen3_5` support entirely. This notebook installs the **latest** vLLM release instead (`v0.25.1` as of writing) to test whether that support has actually stabilized, before committing GPU-hours to a full LoRA fine-tune.

**If this fails**, that's the answer needed to fall back to a more mature backbone (e.g. plain Qwen3, not 3.5) without having spent a training run finding out.

## Step 0: Check CUDA works before installing anything

In [ ]:
!nvidia-smi

try:
    import torch
    print("torch already installed:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    assert torch.cuda.is_available(), "CUDA not available - check nvidia-smi output above and pod health before proceeding."
    print("CUDA check passed - safe to proceed to Step 1.")
except ImportError:
    print("torch not installed yet - will be installed fresh in Step 1. Re-run this cell after Step 1 to confirm CUDA works.")

## Step 1: Install vLLM from its own CUDA-matched wheel index

Pinned to a specific recent release (`0.25.1`, the latest as of writing) rather than the `0.9.2` pin used elsewhere - the whole point of this test is to check whether *current* Qwen3.5 support actually works.

**Not using plain `pip install vllm --torch-backend=auto`** - that hits `vllm-project/vllm#43435` (confirmed, closed "not planned" by vLLM maintainers): the default PyPI vLLM wheel's compiled `_C_stable_libtorch` extension is linked against `libcudart.so.13` regardless of the pod's actual CUDA runtime, because `--torch-backend` only controls which *torch* wheel resolves, not vLLM's own precompiled binary. The actual fix (confirmed in that issue's comments after it was closed): install from vLLM's own CUDA-specific wheel index instead of default PyPI. `cu129` in the URL works for any CUDA 12.x on the pod, not just 12.9 specifically.

In [ ]:
!pip install -U uv
!uv pip install --system --reinstall "vllm==0.25.1" \
    --extra-index-url https://wheels.vllm.ai/0.25.1/cu129 \
    --extra-index-url https://download.pytorch.org/whl/cu129 \
    --index-strategy unsafe-best-match

import torch, vllm
print("torch:", torch.__version__)
print("vllm:", vllm.__version__)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available after install - check the pod."

from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.5.0"), (
    f"torch {torch.__version__} is still too old (needs >=2.5.0 for torch.library.infer_schema, "
    "which vllm.env_override imports at startup). If this still fails after the wheel-index fix, "
    "try restarting the kernel first (a torch already imported in this Python process can't be "
    "swapped out by a pip/uv reinstall alone) and re-running from Step 1."
)

# The actual regression test: importing vllm itself is where the libcudart.so.13 error
# surfaced before (vllm.platforms.cuda imports vllm._C_stable_libtorch at import time).
# If this cell completes without an ImportError, the wheel-index fix worked.
print("\n=== vLLM imported successfully - libcudart issue did not reproduce. ===")

## Step 2: Launch vLLM serving the base Qwen/Qwen3.5-9B (no LoRA, no fine-tuning)

Just testing whether vLLM can load and serve this checkpoint at all. `--trust-remote-code` is included since newer/hybrid architectures sometimes need custom modeling code not yet fully upstreamed into `transformers`.

Also strips a blank `CUDA_VISIBLE_DEVICES` from the launched process's environment - the same Community Cloud quirk documented in `muspsy_serve_vllm.ipynb`: `nvidia-smi` (NVML) reports the GPU as healthy, but an empty-string `CUDA_VISIBLE_DEVICES` hides it from CUDA's own runtime init, causing `RuntimeError: CUDA unknown error` inside the spawned EngineCore worker. Confirmed present on this pod (`echo $CUDA_VISIBLE_DEVICES` printed `[]`).</cell id="cell-5">


In [ ]:
import subprocess, os

BASE_MODEL = "Qwen/Qwen3.5-9B"
API_PORT = 8000
API_KEY = "qwen35-test-key"

serve_cmd = [
    "vllm", "serve", BASE_MODEL,
    "--port", str(API_PORT),
    "--api-key", API_KEY,
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--dtype", "bfloat16",
    "--trust-remote-code",
]
print("Command:", " ".join(serve_cmd))

env = os.environ.copy()
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Community Cloud quirk (confirmed on this pod): CUDA_VISIBLE_DEVICES="" hides the GPU
# from CUDA's runtime init even though nvidia-smi/NVML reports it healthy, causing
# "RuntimeError: CUDA unknown error" inside the spawned EngineCore worker. Strip it
# entirely (rather than setting to "0") so CUDA falls back to its own default device
# enumeration instead of trusting a var that's already proven unreliable on this pod.
if env.get("CUDA_VISIBLE_DEVICES", None) == "":
    print(f"[Fix] CUDA_VISIBLE_DEVICES was set to an empty string - removing it from the launched process's environment.")
    del env["CUDA_VISIBLE_DEVICES"]

log_file = open("qwen35_test_server.log", "w")
server_proc = subprocess.Popen(serve_cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env)
print(f"Server starting in background, PID={server_proc.pid}. Logs: qwen35_test_server.log")

## Step 3: Wait for the server to become ready (or fail with a real error)

This is the actual compatibility test - if `qwen3_5` isn't properly supported, this is where it'll show an architecture-not-recognized or weight-loading error in the log rather than becoming ready.

In [ ]:
import time, requests

url = f"http://localhost:{API_PORT}/v1/models"
headers = {"Authorization": f"Bearer {API_KEY}"}

for attempt in range(60):
    if server_proc.poll() is not None:
        print("=== Server exited early. Log tail: ===")
        with open("qwen35_test_server.log") as f:
            print(f.read()[-4000:])
        raise RuntimeError(f"Server process exited early (code {server_proc.returncode}). See log above.")
    try:
        resp = requests.get(url, headers=headers, timeout=5)
        if resp.status_code == 200:
            print("Server is up.")
            print(resp.json())
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    print("=== Timed out. Log tail: ===")
    with open("qwen35_test_server.log") as f:
        print(f.read()[-4000:])
    raise TimeoutError("Server did not become ready in time. Check the log above - if it's still loading (not erroring), just re-run this cell.")

## Step 4: Sanity-check with a real text-only chat request

In [ ]:
chat_url = f"http://localhost:{API_PORT}/v1/chat/completions"
payload = {
    "model": BASE_MODEL,
    "messages": [
        {"role": "user", "content": "Hi, I've been feeling anxious about an upcoming exam. Can you help?"}
    ],
    "temperature": 0.7,
    "max_tokens": 300,
}
resp = requests.post(chat_url, headers={**headers, "Content-Type": "application/json"}, json=payload, timeout=60)
resp.raise_for_status()
print(resp.json()["choices"][0]["message"]["content"])
print("\n\n=== SUCCESS: Qwen/Qwen3.5-9B serves correctly on this vLLM version. Safe to proceed with fine-tuning. ===")

## Step 5: Stop the test server

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=30)
print("Server stopped.")